# TRM Compiler — Colab Entry Point

Safe Google Colab notebook for the compiler TRM prototype.

Default behavior:
- installs the project
- runs compiler preflight
- runs the compiler pipeline in **synthetic mode**

This avoids assuming `compiler_gym` is available in every Colab image.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path('/content/youtube-videos')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
# Install dependencies - MUST fix setuptools/wheel first to fix gym install
# See: https://stackoverflow.com/questions/76129688
!pip install setuptools==65.5.0 "wheel<0.40.0" -q
!pip uninstall -y trm-experiments >/dev/null 2>&1 || true
!pip install -e .

In [ ]:
# Install compiler_gym (requires old gym which needs fixed setuptools)
!pip install compiler_gym -q

In [ ]:
# Run once after install in Colab, then continue below after restart.
import os
print('Restarting Colab runtime to clear stale imports...')
os.kill(os.getpid(), 9)

In [ ]:
from trm_compiler.env_check import print_preflight
preflight = print_preflight(require_compiler_gym=False)
preflight

In [ ]:
!python -m trm_compiler.colab_smoke

In [ ]:
from pathlib import Path

from torch.utils.data import DataLoader
import torch

from trm_compiler.data import CompilerTraceDataset, generate_compiler_traces
from trm_compiler.model import TinyPassOrderingRefiner, rollout_pass_optimizer
from trm_compiler.training import train_one_epoch
from trm_compiler.baselines import random_search, greedy_search
from trm_compiler.env_wrapper import make_compiler_env

traces = generate_compiler_traces(
    benchmarks=['qsort', 'adpcm', 'sha'],
    episodes_per_benchmark=8,
    max_steps_per_episode=12,
    use_heuristic=True,
    seed=42,
    strategy='mixed',
)
print('num_traces =', len(traces))

dataset = CompilerTraceDataset(traces)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)
model = TinyPassOrderingRefiner()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    losses = train_one_epoch(model, loader, optimizer, 'cpu')
    print(f'epoch={epoch+1}', {k: round(v, 4) for k, v in losses.items()})

env = make_compiler_env('qsort', use_compilergym=False)
obs, _ = env.reset()
trace = rollout_pass_optimizer(model, obs, max_steps=8, temperature=1.0, device='cpu')
print('rollout_steps =', len(trace))
print('first_steps =', trace[:3])

print('random_baseline =', random_search('qsort', max_steps=12, num_trials=10, seed=42))
print('greedy_baseline =', greedy_search('qsort', max_steps=12, seed=42))